In [36]:
# auto reload modules when they change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from src import config
from src.data import compute_rate_constants, load_dataset, select_reactions
from src.kinetics import calculate_rate
from src.perturbation import run_reaction_pipeline_mode2
from src.plotting import plot_kinetics_roh
from src.simulation import exponential_trajectory_points, run_simulation

sns.set_style("whitegrid")
colors = sns.color_palette("muted")
%matplotlib widget

In [38]:
from scipy import constants

hartree_energy_j = constants.physical_constants["Hartree energy"][0]
joules_per_kcal = constants.calorie * constants.kilo

HARTREE_TO_KCAL_MOL = hartree_energy_j * constants.Avogadro / joules_per_kcal

print(HARTREE_TO_KCAL_MOL)
print(f"Hartree to kcal/mol conversion factor: {HARTREE_TO_KCAL_MOL:.4f} kcal/mol per Hartree")

627.5094740628974
Hartree to kcal/mol conversion factor: 627.5095 kcal/mol per Hartree


In [48]:
# Global run controls
RUN_MODE = "mode2"  # "mode1" or "mode2"

# I/O
DATASET_PATH = "imports/calculated-exchange.xlsx"
DATASET_SHEET = "dft_data_tmp"
EXPORT_DIR = Path("exports")
PLOTS_DIR = EXPORT_DIR / "batch-plots"

# Filtering
REACTION_FILTERS = {
    "imido": "im1",
    "roh": ["w1m", "d1m", "0", "w1p", "12", "9", "10", "d2m", "w2m", "8", "4", "1", "5"],
}

# Simulation controls
TEMPERATURE_C = config.TEMPERATURE_DEFAULT
SIMULATION_TIME = config.SIMULATION_TIME_DEFAULT
N_TIME_POINTS = config.TRAJECTORY_POINTS_DEFAULT
TIME_EXPONENT = config.TRAJECTORY_EXPONENT_DEFAULT
INITIAL_CONCENTRATIONS = dict(config.INITIAL_CONCENTRATIONS)

# Mode 2 controls
N_SAMPLES = config.PERTURBATION_N_SAMPLES_DEFAULT
SIGMA = config.PERTURBATION_SIGMA_DEFAULT
RANDOM_SEED = config.PERTURBATION_SEED_DEFAULT

# Optional plotting
SAVE_PLOTS = True
FIXED_PLOT_LIMITS = False

options = {
    "RUN_MODE": RUN_MODE,
    "DATASET_PATH": DATASET_PATH,
    "DATASET_SHEET": DATASET_SHEET,
    "EXPORT_DIR": str(EXPORT_DIR),
    "PLOTS_DIR": str(PLOTS_DIR),
    "REACTION_FILTERS": REACTION_FILTERS,
    "TEMPERATURE_C": TEMPERATURE_C,
    "SIMULATION_TIME": SIMULATION_TIME,
    "N_TIME_POINTS": N_TIME_POINTS,
    "TIME_EXPONENT": TIME_EXPONENT,
    "INITIAL_CONCENTRATIONS": INITIAL_CONCENTRATIONS,
    "N_SAMPLES": N_SAMPLES,
    "SIGMA": SIGMA,
    "RANDOM_SEED": RANDOM_SEED,
    "SAVE_PLOTS": SAVE_PLOTS,
    "FIXED_PLOT_LIMITS": FIXED_PLOT_LIMITS,
}

print("Configured options:\n")
for key, value in options.items():
    print(f"- {key}: {value}")

Configured options:

- RUN_MODE: mode2
- DATASET_PATH: imports/calculated-exchange.xlsx
- DATASET_SHEET: dft_data_tmp
- EXPORT_DIR: exports
- PLOTS_DIR: exports/batch-plots
- REACTION_FILTERS: {'imido': 'im1', 'roh': ['w1m', 'd1m', '0', 'w1p', '12', '9', '10', 'd2m', 'w2m', '8', '4', '1', '5']}
- TEMPERATURE_C: 22
- SIMULATION_TIME: 3600
- N_TIME_POINTS: 1000
- TIME_EXPONENT: 3
- INITIAL_CONCENTRATIONS: {'bispyr': 0.1, 'roh': 0.1, 'me2pyr': 0.0, 'map': 0.0, 'bis': 0.0}
- N_SAMPLES: 1000
- SIGMA: 2.0
- RANDOM_SEED: 42
- SAVE_PLOTS: True
- FIXED_PLOT_LIMITS: False


In [49]:
def calculate_mechanism_roh(energies:pd.Series):
    reactants = energies['bispyrrolide'] + energies['roh'] + energies['roh']
    parsed_energies = {
        "reactants": reactants,
        "a1": energies["a1"] + energies['roh'],
        "ts1": energies["ts1"] + energies['roh'],
        "map": energies["map"] + energies['roh'] + energies['pyrrole'],
        "a2": energies["a2"] + energies['pyrrole'],
        "ts2": energies["ts2"] + energies['pyrrole'],
        "bis": energies["bis"] + energies['pyrrole'] + energies['pyrrole'],
    }
    parsed_energies = pd.Series(parsed_energies)
    return parsed_energies

def calculate_barriers_roh(energies:pd.Series):
    b1 = energies['ts1'] - energies['reactants']
    b2 = energies['ts2'] - energies['map']
    b3 = energies['ts1'] - energies['map']
    b4 = energies['ts2'] - energies['bis']
    return pd.Series({'b1':b1, 'b2':b2, 'b3':b3, 'b4':b4})

calculate_mechanism = calculate_mechanism_roh
calculate_barriers = calculate_barriers_roh

In [50]:
dataset = load_dataset(DATASET_PATH, sheet_name=DATASET_SHEET)
# Replace exact zeros by np.nan
dataset.replace(0, np.nan, inplace=True)
display(dataset)

,bispyrrolide,roh,pyrrole,map,bis,ts1,ts2,a1,a2
im1-ad1,-1555.943152,-538.811186,-288.950229,-1805.831132,-2055.712324,-2094.731577,-2344.621082,-2094.747645,-2344.639115
im1-oc1,-1555.943152,-1651.648709,-288.950229,-2918.667213,-4281.381723,-3207.570314,-4570.287292,-3207.581884,NaN
im1-oc3,-1555.943152,-6600.028793,-288.950229,-7867.044910,-14178.135062,-8155.945641,-14467.033248,-8155.955660,NaN
im2-bitet0,-1398.645500,-1448.284828,-288.950229,-2558.001177,-3717.351588,-2846.909797,-4006.263625,-2846.923488,-4006.276850
im2-oc0,-1398.645500,-1453.069465,-288.950229,-2562.786686,-3726.923637,-2851.692856,-4015.828483,-2851.707342,-4015.840237
im2-oc1,-1398.645500,-1651.648709,-288.950229,-2761.369591,-4124.084736,-3050.267014,-4412.994284,-3050.285522,-4413.014600
im2-oc3,-1398.645500,-6600.028793,-288.950229,-7709.745834,-14020.841722,-7998.648064,-14309.730607,-7998.664104,-14309.760968
im3-j15,-1816.414279,-769.983562,-288.950229,-2297.469619,-2778.529963,-2586.367894,-3067.423651,-2586.379939,-3067.445317
im3-j26,-1816.414279,-1005.967348,-288.950229,-2533.456775,-3250.481085,-2822.345999,-3539.362038,-2822.358298,NaN
im3-j29,-1816.414279,-1232.330923,-288.950229,-2759.819679,-3703.222974,-3048.713639,-3992.115821,-3048.726887,NaN


In [51]:
mechanism_df = dataset.T.apply(calculate_mechanism)

mechanism_df -= mechanism_df.loc['reactants']  # Set reactants as zero reference
mechanism_df *= HARTREE_TO_KCAL_MOL  # Convert from Hartree to kcal/mol
mechanism_df = mechanism_df.T

display(mechanism_df)

barriers_df = mechanism_df.T.apply(calculate_barriers).T
display(barriers_df)

# Convert barriers -> rate constants using phase-1 utilities
rates_df = compute_rate_constants(barriers_df, temperature=TEMPERATURE_C)
display(rates_df.head())

# selected_rates = select_reactions(rates_df, REACTION_FILTERS)

for i, idx in enumerate(rates_df.index):
    print(f"{i}: {idx}")

selected_rates = rates_df.copy()
selected_rates = selected_rates.iloc[[8]]
print(f"Selected reactions: {len(selected_rates)}")
display(selected_rates.head())

,reactants,a1,ts1,map,a2,ts2,bis
im1-ad1,0.0,4.199944,14.282870,-16.957219,-14.946953,-3.631327,-29.654451
im1-oc1,0.0,6.260683,13.520900,-16.052217,NaN,1.913331,-26.111676
im1-oc3,0.0,10.219100,16.505850,-14.554480,NaN,10.831242,-21.826549
im2-bitet0,0.0,4.292296,12.883109,-13.226644,-7.481890,0.816772,-23.149279
im2-oc0,0.0,4.783724,13.873443,-13.773849,-3.787370,3.588213,-24.890071
im2-oc1,0.0,5.450769,17.064678,-16.071304,-13.749889,-1.001493,-26.529116
im2-oc3,0.0,6.393561,16.458953,-13.661200,-5.090100,13.961707,-24.532579
im3-j15,0.0,11.234260,18.792093,-13.809316,-8.874328,4.720889,-30.759232
im3-j26,0.0,14.639597,22.357528,-15.923986,NaN,23.034963,-20.436313
im3-j29,0.0,11.493158,19.805992,-15.503004,NaN,6.322115,-29.686251


,b1,b2,b3,b4
im1-ad1,14.282870,13.325892,31.240090,26.023124
im1-oc1,13.520900,17.965548,29.573118,28.025007
im1-oc3,16.505850,25.385722,31.060330,32.657791
im2-bitet0,12.883109,14.043415,26.109753,23.966051
im2-oc0,13.873443,17.362062,27.647291,28.478284
im2-oc1,17.064678,15.069811,33.135982,25.527623
im2-oc3,16.458953,27.622907,30.120153,38.494286
im3-j15,18.792093,18.530206,32.601409,35.480122
im3-j26,22.357528,38.958948,38.281514,43.471276
im3-j29,19.805992,21.825119,35.308995,36.008366


,k1d,k1r,k2d,k2r
im1-ad1,163.098741,4.525682e-11,8.338530e+02,3.302406e-07
im1-oc1,597.980138,7.763748e-10,3.058015e-01,1.087533e-08
im1-oc3,3.684183,6.148869e-11,9.790939e-07,4.035345e-12
im2-bitet0,1774.063605,2.848931e-07,2.453387e+02,1.101759e-05
im2-oc0,327.816072,2.070806e-08,8.556948e-01,5.021024e-09


0: im1-ad1
1: im1-oc1
2: im1-oc3
3: im2-bitet0
4: im2-oc0
5: im2-oc1
6: im2-oc3
7: im3-j15
8: im3-j26
9: im3-j29
10: im3-j31
11: im4-oc1
12: im5-oc1
13: im5-oc2
14: im5-oc3
15: im5-oc4
Selected reactions: 1


,k1d,k1r,k2d,k2r
im3-j26,0.000171,2.763960e-16,8.707508e-17,3.967619e-20


In [52]:
# Quick run reminder:
# 1) Set RUN_MODE = "mode1" for baseline from rates.
# 2) Set RUN_MODE = "mode2" and implement mechanism_energies_for_mode2 + run_single_pipeline_mode2.
# 3) Re-run notebook top to bottom.

print(f"Configured run mode: {RUN_MODE}")
print(f"Selected reactions: {len(selected_rates)}")
print("Ready to execute.")

Configured run mode: mode2
Selected reactions: 1
Ready to execute.


In [53]:
def simulate_from_rates(reaction_rates: pd.Series) -> tuple[pd.DataFrame, dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    points = exponential_trajectory_points(SIMULATION_TIME, num_points=N_TIME_POINTS, exponent=TIME_EXPONENT)
    simulation = run_simulation(reaction_rates.to_dict(), INITIAL_CONCENTRATIONS, points)

    concentrations = {
        label: simulation["y"][idx]
        for idx, label in enumerate(config.ENTITIES)
    }

    c0 = concentrations["bispyr"][0]
    yields = {k: (v / c0) for k, v in concentrations.items()}
    conversions = {k: (1 - v) for k, v in yields.items()}
    conversions["roh"] = 1 - (concentrations["roh"] / concentrations["roh"][0])

    # NOTICE:
    # We use the yields (not concentrations) for the trajectory DataFrame, to have a common scale (concentration independent scale)
    # But it is equivalent to use concentrations, as they are just scaled by a constant factor (initial concentration of bispyr). 
    trajectory = pd.concat(
        [pd.Series(simulation["t"], name="time"), pd.DataFrame(yields)],
        axis=1,
    )
    return trajectory, concentrations, yields, conversions

In [54]:
# Outputs for Mode 1
concentrations_all: dict[str, dict[str, np.ndarray]] = {}
yields_all: dict[str, dict[str, np.ndarray]] = {}
conversions_all: dict[str, dict[str, np.ndarray]] = {}
trajectories_all: dict[str, pd.DataFrame] = {}
summary_rows: list[dict[str, float | str]] = []

if RUN_MODE == "mode1":
    if SAVE_PLOTS:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    for reaction_label, reaction_rates in selected_rates.iterrows():
        print(f"Current reaction: {reaction_label}")

        if reaction_rates.isna().any():
            print("At least one rate constant for this reaction is missing. Skipping simulation")
            continue

        rate_threshold = calculate_rate(25, TEMPERATURE_C)
        if reaction_rates["k1d"] <= rate_threshold:
            print("Warning: 1st exchange may be inaccessible")

        trajectory, concentrations, yields, conversions = simulate_from_rates(reaction_rates)

        xt = len(trajectory) - 1
        summary_rows.append(
            {
                "reaction": reaction_label,
                "conv_bispyr_final": float(conversions["bispyr"][xt]),
                "conv_roh_final": float(conversions["roh"][xt]),
                "yield_map_final": float(yields["map"][xt]),
                "yield_bis_final": float(yields["bis"][xt]),
            }
        )

        concentrations_all[reaction_label] = concentrations
        yields_all[reaction_label] = yields
        conversions_all[reaction_label] = conversions
        trajectories_all[reaction_label] = trajectory

        if SAVE_PLOTS:
            plot_kinetics_roh(
                times=trajectory["time"].to_numpy(),
                concentrations=concentrations,
                reaction_label=reaction_label,
                colors=colors,
                set_ylim=FIXED_PLOT_LIMITS,
                set_xlim=FIXED_PLOT_LIMITS,
                save_path=str(PLOTS_DIR),
            )

    mode1_summary = pd.DataFrame(summary_rows)
    display(mode1_summary.head())

In [55]:
def run_single_pipeline_mode2(perturbed_mechanism: pd.Series) -> pd.DataFrame:
    """Run one deterministic pipeline instance for a perturbed mechanism series."""
    barriers = calculate_barriers(perturbed_mechanism)
    rates_one = compute_rate_constants(pd.DataFrame([barriers]), temperature=TEMPERATURE_C).iloc[0]
    trajectory, concentrations, yields, conversions = simulate_from_rates(rates_one)
    return trajectory

mode2_results: dict[str, dict[str, object]] = {}

if RUN_MODE == "mode2":
    for reaction_label in tqdm(selected_rates.index, desc="Mode 2 reactions"):
        print(f"Current reaction: {reaction_label}")

        baseline_mechanism = mechanism_df.loc[reaction_label]
        mode2_results[reaction_label] = run_reaction_pipeline_mode2(
            baseline_energies=baseline_mechanism,
            run_single_pipeline=run_single_pipeline_mode2,
            n_samples=1000,
            sigma=SIGMA,
            random_seed=RANDOM_SEED,
            aggregate=False,
            export_trajectories=True,
            export_dir=EXPORT_DIR,
            reaction_label=reaction_label,
        )

    print(f"Mode 2 completed for {len(mode2_results)} reactions")

Mode 2 reactions:   0%|          | 0/1 [00:00<?, ?it/s]

Current reaction: im3-j26
Mode 2 completed for 1 reactions


In [56]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "mode1":
    mode1_summary.to_csv(EXPORT_DIR / "mode1_summary.csv", index=False)

    # Optional trajectory export (long format)
    mode1_long = []
    for reaction_label, traj_df in trajectories_all.items():
        tmp = traj_df.copy()
        tmp.insert(0, "reaction", reaction_label)
        mode1_long.append(tmp)

    if mode1_long:
        pd.concat(mode1_long, ignore_index=True).to_csv(EXPORT_DIR / "mode1_trajectories.csv", index=False)

    print("Exported Mode 1 summary and trajectories")

if RUN_MODE == "mode2":
    for reaction_label, result in tqdm(mode2_results.items(), desc="Mode 2 exports", total=len(mode2_results)):
        result["samples"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_samples.csv", index=False)

        # Avoid duplicate writes: if core pipeline already exported perturbed energies,
        # do not export the same table again from this notebook cell.
        if "perturbed_energies_export_path" not in result:
            perturbed_df = result.get("perturbed_energies")
            if perturbed_df is not None:
                perturbed_df.to_csv(
                    EXPORT_DIR / f"{reaction_label}_mode2_perturbed_energies.csv", index=False
                )
            else:
                print(
                    f"Warning: perturbed_energies missing for {reaction_label}. "
                    "Re-run the Mode 2 cell to regenerate full perturbation tracking."
                )

        agg = result.get("aggregate")
        if agg:
            agg["mean"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_mean.csv", index=False)
            agg["std"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_std.csv", index=False)
            agg["percentiles"].to_csv(EXPORT_DIR / f"{reaction_label}_mode2_percentiles.csv", index=False)

        if "trajectories_export_path" in result:
            print(f"Trajectories exported: {result['trajectories_export_path']}")
        if "perturbed_energies_export_path" in result:
            print(f"Perturbed energies exported: {result['perturbed_energies_export_path']}")

    print("Exported Mode 2 samples, perturbed energies, and aggregate tables")

Mode 2 exports:   0%|          | 0/1 [00:00<?, ?it/s]

Trajectories exported: exports/im3-j26_mode2_trajectories.pkl.gz
Perturbed energies exported: exports/im3-j26_mode2_perturbed_energies.csv
Exported Mode 2 samples, perturbed energies, and aggregate tables
